# MLL Project Assignment

## Business Understanding

## Data understanding

In [15]:

import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from IPython.display import display, Markdown
from surprise import Reader, Dataset, KNNBasic, accuracy
from surprise.model_selection import train_test_split, cross_validate

In [16]:
# Load the dataset
movies = pd.read_csv('datasets/ml-latest-small/movies.csv', sep=',')
ratings = pd.read_csv('datasets/ml-latest-small/ratings.csv', sep=',')
from surprise import Reader, Dataset, KNNBasic, accuracy
from surprise.model_selection import train_test_split, cross_validate

overview = pd.DataFrame({
    "dataset": ["movies", "ratings"],
    "rows": [movies.shape[0], ratings.shape[0]],
    "columns": [movies.shape[1], ratings.shape[1]]
})

display(overview.style.hide(axis="index"))


movies_columns = pd.DataFrame({
    "column": movies.columns,
    "data_type": movies.dtypes.astype(str).values
})
display(movies_columns.style.hide(axis="index"))

ratings_columns = pd.DataFrame({
    "column": ratings.columns,
    "data_type": ratings.dtypes.astype(str).values
})
display(ratings_columns.style.hide(axis="index"))


display(movies.head(10).style.hide(axis="index"))


display(ratings.head(10).style.hide(axis="index"))



dataset,rows,columns
movies,9742,3
ratings,100836,4


column,data_type
movieId,int64
title,str
genres,str


column,data_type
userId,int64
movieId,int64
rating,float64
timestamp,int64


movieId,title,genres
1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,Jumanji (1995),Adventure|Children|Fantasy
3,Grumpier Old Men (1995),Comedy|Romance
4,Waiting to Exhale (1995),Comedy|Drama|Romance
5,Father of the Bride Part II (1995),Comedy
6,Heat (1995),Action|Crime|Thriller
7,Sabrina (1995),Comedy|Romance
8,Tom and Huck (1995),Adventure|Children
9,Sudden Death (1995),Action
10,GoldenEye (1995),Action|Adventure|Thriller


userId,movieId,rating,timestamp
1,1,4.000000,964982703
1,3,4.000000,964981247
1,6,4.000000,964982224
1,47,5.000000,964983815
1,50,5.000000,964982931
1,70,3.000000,964982400
1,101,5.000000,964980868
1,110,4.000000,964982176
1,151,5.000000,964984041
1,157,5.000000,964984100


## Data preparation

In [17]:
# Create copies so the raw datasets remain untouched
movies_prepared = movies.copy()
ratings_prepared = ratings.copy()


# Extract year from title
year_str = movies_prepared['title'].str.extract(r'\((\d{4})\)', expand=False)

# Remove year from title text
movies_prepared['title'] = movies_prepared['title'].str.replace(r'\s*\(\d{4}\)', '', regex=True)

# Create separate year column
movies_prepared['year'] = pd.to_numeric(year_str, errors='coerce').astype('Int64')

year_summary = pd.DataFrame({
    "action": [
        "Year extracted from title",
        "Title cleaned",
        "Year column created"
    ],
    "status": ["Done", "Done", "Done"]
})
display(year_summary.style.hide(axis="index"))


movie_ratings = pd.merge(ratings_prepared, movies_prepared, on='movieId')

merge_summary = pd.DataFrame({
    "metric": ["Rows after merge", "Columns after merge"],
    "value": [movie_ratings.shape[0], movie_ratings.shape[1]]
})
display(merge_summary.style.hide(axis="index"))


null_table = movie_ratings.isnull().sum().reset_index()
null_table.columns = ["column", "missing_values"]
display(null_table.style.hide(axis="index"))

total_missing = int(null_table["missing_values"].sum())

missing_summary = pd.DataFrame({
    "metric": ["Total missing values"],
    "value": [total_missing]
})
display(missing_summary.style.hide(axis="index"))

# Drop rows only if important fields are missing
movie_ratings = movie_ratings.dropna(subset=['userId', 'movieId', 'rating'])


dtype_table = movie_ratings.dtypes.astype(str).reset_index()
dtype_table.columns = ["column", "data_type"]
display(dtype_table.style.hide(axis="index"))


duplicates = int(movie_ratings.duplicated().sum())

duplicate_summary = pd.DataFrame({
    "metric": ["Duplicate rows before cleaning"],
    "value": [duplicates]
})
display(duplicate_summary.style.hide(axis="index"))

if duplicates > 0:
    movie_ratings = movie_ratings.drop_duplicates()


movie_ratings['timestamp'] = pd.to_datetime(movie_ratings['timestamp'], unit='s')

timestamp_summary = pd.DataFrame({
    "action": ["Timestamp converted to datetime"],
    "status": ["Done"]
})
display(timestamp_summary.style.hide(axis="index"))

display(movie_ratings.head(10).style.hide(axis="index"))

action,status
Year extracted from title,Done
Title cleaned,Done
Year column created,Done


metric,value
Rows after merge,100836
Columns after merge,7


column,missing_values
userId,0
movieId,0
rating,0
timestamp,0
title,0
genres,0
year,18


metric,value
Total missing values,18


column,data_type
userId,int64
movieId,int64
rating,float64
timestamp,int64
title,str
genres,str
year,Int64


metric,value
Duplicate rows before cleaning,0


action,status
Timestamp converted to datetime,Done


userId,movieId,rating,timestamp,title,genres,year
1,1,4.000000,2000-07-30 18:45:03,Toy Story,Adventure|Animation|Children|Comedy|Fantasy,1995
1,3,4.000000,2000-07-30 18:20:47,Grumpier Old Men,Comedy|Romance,1995
1,6,4.000000,2000-07-30 18:37:04,Heat,Action|Crime|Thriller,1995
1,47,5.000000,2000-07-30 19:03:35,Seven (a.k.a. Se7en),Mystery|Thriller,1995
1,50,5.000000,2000-07-30 18:48:51,"Usual Suspects, The",Crime|Mystery|Thriller,1995
1,70,3.000000,2000-07-30 18:40:00,From Dusk Till Dawn,Action|Comedy|Horror|Thriller,1996
1,101,5.000000,2000-07-30 18:14:28,Bottle Rocket,Adventure|Comedy|Crime|Romance,1996
1,110,4.000000,2000-07-30 18:36:16,Braveheart,Action|Drama|War,1995
1,151,5.000000,2000-07-30 19:07:21,Rob Roy,Action|Drama|Romance|War,1995
1,157,5.000000,2000-07-30 19:08:20,Canadian Bacon,Comedy|War,1995


## Modeling

In [18]:

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(movie_ratings[['userId', 'movieId', 'rating']], reader)

sim_options = {
    'name': 'cosine',
    'user_based': True
}

model_setup = pd.DataFrame({
    "setting": [
        "Algorithm",
        "Collaborative filtering type",
        "Similarity metric",
        "Rating scale"
    ],
    "value": [
        "KNNBasic",
        "User-based",
        "Cosine",
        "0.5 to 5.0"
    ]
})

display(model_setup.style.hide(axis="index"))


trainset_eval, testset_eval = train_test_split(data, test_size=0.2, random_state=42)

eval_model = KNNBasic(sim_options=sim_options)
eval_model.fit(trainset_eval)
eval_predictions = eval_model.test(testset_eval)

holdout_rmse = accuracy.rmse(eval_predictions, verbose=False)
holdout_mae = accuracy.mae(eval_predictions, verbose=False)

holdout_results = pd.DataFrame({
    "metric": ["RMSE", "MAE", "Train size", "Test size"],
    "value": [
        round(holdout_rmse, 4),
        round(holdout_mae, 4),
        trainset_eval.n_ratings,
        len(testset_eval)
    ]
})

display(holdout_results.style.hide(axis="index"))


cv_model = KNNBasic(sim_options=sim_options)
cv_results = cross_validate(
    cv_model,
    data,
    measures=["RMSE", "MAE"],
    cv=3,
    verbose=False
)

num_folds = len(cv_results["test_rmse"])

cv_table = pd.DataFrame({
    "fold": [f"Fold {i+1}" for i in range(num_folds)],
    "RMSE": np.round(cv_results["test_rmse"], 4),
    "MAE": np.round(cv_results["test_mae"], 4),
    "Fit time (s)": np.round(cv_results["fit_time"], 2),
    "Test time (s)": np.round(cv_results["test_time"], 2)
})

cv_mean = pd.DataFrame({
    "fold": ["Mean"],
    "RMSE": [round(np.mean(cv_results["test_rmse"]), 4)],
    "MAE": [round(np.mean(cv_results["test_mae"]), 4)],
    "Fit time (s)": [round(np.mean(cv_results["fit_time"]), 2)],
    "Test time (s)": [round(np.mean(cv_results["test_time"]), 2)]
})

cv_display = pd.concat([cv_table, cv_mean], ignore_index=True)
display(cv_display.style.hide(axis="index"))


final_trainset = data.build_full_trainset()
final_model = KNNBasic(sim_options=sim_options)
final_model.fit(final_trainset)

final_model_summary = pd.DataFrame({
    "status": ["Final model trained successfully"],
    "training_rows": [movie_ratings.shape[0]],
    "unique_users": [movie_ratings['userId'].nunique()],
    "unique_movies": [movie_ratings['movieId'].nunique()]
})

display(final_model_summary.style.hide(axis="index"))


def get_top_n_recommendations(user_id, n=10):
    rated_movies = movie_ratings[movie_ratings['userId'] == user_id]['movieId'].unique()
    all_movies = movies_prepared['movieId'].unique()
    unrated_movies = [movie_id for movie_id in all_movies if movie_id not in rated_movies]

    predictions = []
    for movie_id in unrated_movies:
        pred = final_model.predict(user_id, movie_id)
        movie_info = movies_prepared[movies_prepared['movieId'] == movie_id].iloc[0]
        predictions.append({
            "movieId": movie_id,
            "title": movie_info['title'],
            "genres": movie_info['genres'],
            "predicted_rating": round(pred.est, 2)
        })

    recommendations = pd.DataFrame(predictions)
    recommendations = recommendations.sort_values(by='predicted_rating', ascending=False).head(n)
    return recommendations

display(pd.DataFrame({
    "helper_function": ["get_top_n_recommendations(user_id, n=10)"],
    "purpose": ["Returns the top-N predicted movies for a selected user"]
}).style.hide(axis="index"))

setting,value
Algorithm,KNNBasic
Collaborative filtering type,User-based
Similarity metric,Cosine
Rating scale,0.5 to 5.0


Computing the cosine similarity matrix...
Done computing similarity matrix.


metric,value
RMSE,0.982300
MAE,0.755900
Train size,80668.000000
Test size,20168.000000


Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.
Computing the cosine similarity matrix...
Done computing similarity matrix.


fold,RMSE,MAE,Fit time (s),Test time (s)
Fold 1,0.969200,0.748300,0.080000,1.400000
Fold 2,0.980800,0.757100,0.080000,1.150000
Fold 3,0.983200,0.756800,0.090000,1.200000
Mean,0.977700,0.754100,0.080000,1.250000


Computing the cosine similarity matrix...
Done computing similarity matrix.


status,training_rows,unique_users,unique_movies
Final model trained successfully,100836,610,9724


helper_function,purpose
"get_top_n_recommendations(user_id, n=10)",Returns the top-N predicted movies for a selected user


## Evaluation

In [19]:

evaluation_summary = pd.DataFrame({
    "evaluation_method": ["Holdout split", "3-fold cross-validation"],
    "RMSE": [
        round(holdout_rmse, 4),
        round(np.mean(cv_results["test_rmse"]), 4)
    ],
    "MAE": [
        round(holdout_mae, 4),
        round(np.mean(cv_results["test_mae"]), 4)
    ]
})

display(evaluation_summary.style.hide(axis="index"))


interpretation_table = pd.DataFrame({
    "metric": ["RMSE", "MAE"],
    "meaning": [
        "Measures prediction error and gives more weight to larger mistakes",
        "Measures the average absolute difference between predicted and real ratings"
    ],
    "interpretation": [
        "Lower RMSE means the recommender predicts user ratings more accurately",
        "Lower MAE means the average prediction error is smaller"
    ]
})

display(interpretation_table.style.hide(axis="index"))


num_users = movie_ratings['userId'].nunique()
num_movies = movie_ratings['movieId'].nunique()
num_ratings = movie_ratings.shape[0]
possible_user_movie_pairs = num_users * num_movies
sparsity = 1 - (num_ratings / possible_user_movie_pairs)

context_table = pd.DataFrame({
    "metric": [
        "Unique users",
        "Unique movies",
        "Observed ratings",
        "Possible user-movie pairs",
        "Dataset sparsity"
    ],
    "value": [
        num_users,
        num_movies,
        num_ratings,
        possible_user_movie_pairs,
        f"{sparsity:.2%}"
    ]
})

display(context_table.style.hide(axis="index"))


evaluation_notes = pd.DataFrame({
    "aspect": [
        "Strength",
        "Strength",
        "Limitation",
        "Limitation"
    ],
    "description": [
        "The model can generate personalized recommendations from historical rating behavior.",
        "Cross-validation gives a more reliable estimate than a single train-test split.",
        "The model has a cold-start problem for new users and new movies with little or no rating history.",
        "Recommendations are based on rating patterns only and do not use movie content such as plot or actors."
    ]
})

display(evaluation_notes.style.hide(axis="index"))



evaluation_method,RMSE,MAE
Holdout split,0.982300,0.755900
3-fold cross-validation,0.977700,0.754100


metric,meaning,interpretation
RMSE,Measures prediction error and gives more weight to larger mistakes,Lower RMSE means the recommender predicts user ratings more accurately
MAE,Measures the average absolute difference between predicted and real ratings,Lower MAE means the average prediction error is smaller


metric,value
Unique users,610
Unique movies,9724
Observed ratings,100836
Possible user-movie pairs,5931640
Dataset sparsity,98.30%


aspect,description
Strength,The model can generate personalized recommendations from historical rating behavior.
Strength,Cross-validation gives a more reliable estimate than a single train-test split.
Limitation,The model has a cold-start problem for new users and new movies with little or no rating history.
Limitation,Recommendations are based on rating patterns only and do not use movie content such as plot or actors.


## Deployment